In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [6]:
raw_data_path = Path("data/raw")

In [18]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")
tables = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

In [19]:
print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("order_payments:", order_payments.shape)
print("order_reviews:", order_reviews.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("geolocation:", geolocation.shape)
print("category_translation:", category_translation.shape)

customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
geolocation: (1000163, 5)
category_translation: (71, 2)


Получается так, что каждому заказу соотвествует отдельный customer_id

In [20]:
print("customers:", list(customers.columns))
print("orders:", list(orders.columns))
print("order_items:", list(order_items.columns))
print("order_payments:", list(order_payments.columns))
print("order_reviews:", list(order_reviews.columns))
print("products:", list(products.columns))
print("sellers:", list(sellers.columns))
print("geolocation:", list(geolocation.columns))
print("category_translation:", list(category_translation.columns))

customers: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
orders: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
order_items: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
order_payments: ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
order_reviews: ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
products: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
sellers: ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
geolocation: ['ge

- `order_id` связывает `orders`, `order_items`, `order_payments` и `order_reviews`;

- `customer_id` связывает `orders` и `customers`;

- `product_id` связывает `order_items` и `products`;

- `seller_id` связывает `order_items` и `sellers`.

In [21]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [23]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for i in date_columns:
    orders[i] = pd.to_datetime(orders[i])
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [26]:
orders["order_status"].value_counts(normalize=True).round(4) * 100

order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64

Будем брать только со статусов "delivered"

In [27]:
delivered_orders = orders[orders["order_status"] == "delivered"]
delivered_orders.shape

(96478, 8)

In [29]:
missing_summary = []

for table_name, df in tables.items():
    missing_count = df.isna().sum()
    missing_percent = (df.isna().mean() * 100).round(2)

    table_missing = pd.DataFrame({
        "table": table_name,
        "column": missing_count.index,
        "missing_count": missing_count.values,
        "missing_percent": missing_percent.values
    })

    missing_summary.append(table_missing)

missing_summary = pd.concat(missing_summary, ignore_index=True)

missing_summary[missing_summary["missing_count"] > 0].sort_values(
    by=["table", "missing_percent"],
    ascending=[True, False]
)

,table,column,missing_count,missing_percent
25,order_reviews,review_comment_title,87656,88.34
26,order_reviews,review_comment_message,58247,58.70
35,orders,order_delivered_customer_date,2965,2.98
34,orders,order_delivered_carrier_date,1783,1.79
33,orders,order_approved_at,160,0.16
38,products,product_category_name,610,1.85
39,products,product_name_lenght,610,1.85
40,products,product_description_lenght,610,1.85
41,products,product_photos_qty,610,1.85
42,products,product_weight_g,2,0.01



В таблицах присутствуют пропуски, однако часть из них имеет бизнес-объяснение. у недоставленных или незавершённых заказов могут отсутствовать даты фактической доставки. в отзывах могут отсутствовать текстовые комментарии, так как клиент мог оставить только оценку.

In [30]:
important_delivery_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

delivered_orders[important_delivery_columns].isna().sum()

order_purchase_timestamp         0
order_delivered_customer_date    8
order_estimated_delivery_date    0
dtype: int64

In [31]:
duplicates_summary = []

for table_name, df in tables.items():
    duplicates_summary.append({
        "table": table_name,
        "duplicates_count": df.duplicated().sum()
    })

duplicates_summary = pd.DataFrame(duplicates_summary)
duplicates_summary

,table,duplicates_count
0,customers,0
1,geolocation,261831
2,order_items,0
3,order_payments,0
4,order_reviews,0
5,orders,0
6,products,0
7,sellers,0
8,category_translation,0


Дубликаты только в геолокации

In [32]:
key_uniqueness = pd.DataFrame({
    "table": ["orders", "customers", "products", "sellers"],
    "key": ["order_id", "customer_id", "product_id", "seller_id"],
    "rows": [
        orders.shape[0],
        customers.shape[0],
        products.shape[0],
        sellers.shape[0]
    ],
    "unique_keys": [
        orders["order_id"].nunique(),
        customers["customer_id"].nunique(),
        products["product_id"].nunique(),
        sellers["seller_id"].nunique()
    ]
})

key_uniqueness["is_unique"] = key_uniqueness["rows"] == key_uniqueness["unique_keys"]

key_uniqueness

,table,key,rows,unique_keys,is_unique
0,orders,order_id,99441,99441,True
1,customers,customer_id,99441,99441,True
2,products,product_id,32951,32951,True
3,sellers,seller_id,3095,3095,True


In [33]:
relationship_checks = pd.DataFrame({
    "check": [
        "order_items.order_id in orders.order_id",
        "order_payments.order_id in orders.order_id",
        "order_reviews.order_id in orders.order_id",
        "orders.customer_id in customers.customer_id",
        "order_items.product_id in products.product_id",
        "order_items.seller_id in sellers.seller_id"
    ],
    "missing_keys": [
        (~order_items["order_id"].isin(orders["order_id"])).sum(),
        (~order_payments["order_id"].isin(orders["order_id"])).sum(),
        (~order_reviews["order_id"].isin(orders["order_id"])).sum(),
        (~orders["customer_id"].isin(customers["customer_id"])).sum(),
        (~order_items["product_id"].isin(products["product_id"])).sum(),
        (~order_items["seller_id"].isin(sellers["seller_id"])).sum()
    ]
})

relationship_checks

,check,missing_keys
0,order_items.order_id in orders.order_id,0
1,order_payments.order_id in orders.order_id,0
2,order_reviews.order_id in orders.order_id,0
3,orders.customer_id in customers.customer_id,0
4,order_items.product_id in products.product_id,0
5,order_items.seller_id in sellers.seller_id,0


все ключи из дочерней таблицы найдены в основной таблице.

In [34]:
delivered_orders = delivered_orders.copy()

delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.days

delivered_orders["delay_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.days

delivered_orders["is_late"] = delivered_orders["delay_days"] > 0

delivered_orders[[
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
    "delay_days",
    "is_late"
]].head()

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,False
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,False
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,False
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,False


In [35]:
delivered_orders[["delivery_days", "delay_days"]].describe()

,delivery_days,delay_days
count,96470.000000,96470.000000
mean,12.093604,-11.875889
std,9.551380,10.182105
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-12.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


In [36]:
late_delivery_rate = delivered_orders["is_late"].mean() * 100

round(late_delivery_rate, 2)

np.float64(6.77)

## Итоги проверки качества данных

В ходе первичной проверки качества данных были получены следующие результаты:

- В таблице доставленных заказов почти все ключевые даты заполнены. У 8 доставленных заказов отсутствует фактическая дата доставки клиенту (`order_delivered_customer_date`). Эти строки следует исключить при расчёте сроков доставки.
- Основные ключи в таблицах `orders`, `customers`, `products` и `sellers` уникальны, поэтому их можно использовать для корректного объединения данных.
- Проверка связей между таблицами показала, что ключи из дочерних таблиц присутствуют в основных таблицах. Это позволяет безопасно объединять данные по `order_id`, `customer_id`, `product_id` и `seller_id`.
- Полные дубликаты обнаружены только в таблице `geolocation`. Это ожидаемо, так как один zip-code prefix может иметь несколько записей с координатами. Перед географическим анализом эту таблицу потребуется агрегировать.
- Средний срок доставки среди доставленных заказов составляет около 12 дней.
- В среднем заказы доставлялись примерно на 12 дней раньше ожидаемой даты доставки.
- Доля заказов, доставленных позже ожидаемой даты, составляет около 6.8%.

Таким образом, данные подходят для дальнейшего анализа выручки, доставки, отзывов и клиентского опыта. Основное ограничение: для расчёта сроков доставки необходимо исключить небольшое количество доставленных заказов с пропущенной фактической датой доставки.

In [37]:
delivered_orders_clean = delivered_orders.dropna(
    subset=["order_delivered_customer_date"]
)

delivered_orders_clean.shape

(96470, 11)

Заказы со статусом `delivered`, но без фактической даты доставки, были исключены из анализа доставки. Таких строк всего 8, поэтому их удаление не повлияет существенно на результаты, но позволит корректно рассчитывать сроки доставки и задержки.